# Workflow: branching and parallel steps

This is notebook 2 of 3 in the workflow series. It builds on
[Notebook 1](1_workflow_intro.ipynb), which introduced linear workflows.

The workflow container itself is just an ordered list of step IRIs
(no branching metadata of its own).  **Branching** is expressed at the
step level: each x-process-step schema instance can declare which other
steps precede it via `preceded_by`.

This means the workflow node is a simple ordered envelope, while the
actual dependency graph lives in the step records:

```
sample-prep-1  (pmdco:ManufacturingProcess)
      |  bfo:preceded_by ←─┐
      +──► tensile-test-1  │   ← branch A
      |    (pmdco:TensileTestingProcess)    │
      |                    │
      +──► chem-comp-1     │   ← branch B (parallel)
           (pmdco:TensileTestingProcess)    │
                           │
           calibration-1 ──┘   ← rejoins after tensile test
           (obi:ComputerSimulation)
```

The workflow node simply lists all four step IRIs in a sensible order;
the dependency graph is reconstructed from the step records.

---

## Environment setup

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv && source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE     = pathlib.Path().resolve()   # schemas/workflow/PMDCo/docs/
WORKFLOW = HERE.parent                # schemas/workflow/PMDCo/

schema = Schema(WORKFLOW)

BASE_IRI = "https://example.org/"

## Part A: Build the workflow container

The workflow lists all step IRIs in execution order.  The `steps` array
expresses the _intended sequence_; the actual dependency structure is
declared in the individual step records (Part B).

In [3]:
workflow_input = {
    "label": "Parallel QA workflow, batch A",
    "description": "Parallel QA: two analyses run simultaneously after sample preparation.",
    "steps": [
        BASE_IRI + "steps/sample-prep-1",
        BASE_IRI + "steps/tensile-test-1",
        BASE_IRI + "steps/chem-composition-1",
        BASE_IRI + "steps/model-calibration-1",
    ],
}

wf_graph = schema.to_graph(workflow_input, base=BASE_IRI)
print(f"Workflow graph: {len(wf_graph)} triples")

conforms, violations = schema.validate(wf_graph)
print(f"SHACL conforms: {conforms}")

Workflow graph: 13 triples


SHACL conforms: True


## Part B: Build the step records

Each step is described using its own x-process-step schema.  The
`preceded_by` field on each step record expresses the dependency:

- `tensile-test-1` and `chem-composition-1` both follow `sample-prep-1` (fan-out)
- `model-calibration-1` follows `tensile-test-1` only

In a real deployment each step record would be stored and retrieved
independently.  Here we inline them as Turtle to keep the example
self-contained.

In [4]:
STEP_TTL = """
@prefix bfo:   <http://purl.obolibrary.org/obo/BFO_> .
@prefix rdfs:  <http://www.w3.org/2000/01/rdf-schema#> .
@prefix obi:   <http://purl.obolibrary.org/obo/OBI_> .
@prefix pmdco: <https://w3id.org/pmd/co/> .

<https://example.org/steps/sample-prep-1>
    a pmdco:PMD_0000029 ;
    rdfs:label "Sample Preparation, batch A" .

<https://example.org/steps/tensile-test-1>
    a pmdco:PMD_0000974 ;
    rdfs:label "Tensile Test, batch A" ;
    bfo:0000062 <https://example.org/steps/sample-prep-1> .

<https://example.org/steps/chem-composition-1>
    a pmdco:PMD_0000974 ;
    rdfs:label "Chemical Composition, batch A" ;
    bfo:0000062 <https://example.org/steps/sample-prep-1> .

<https://example.org/steps/model-calibration-1>
    a obi:0000471 ;
    rdfs:label "Hockett-Sherby Calibration, batch A" ;
    bfo:0000062 <https://example.org/steps/tensile-test-1> .
"""

step_graph = rdflib.Graph()
step_graph.parse(data=STEP_TTL, format="turtle")
print(f"Step graph: {len(step_graph)} triples")

Step graph: 11 triples


## Part C: Combine and query the full dependency graph

Merging the workflow graph and the step graphs gives a combined view.
SPARQL then reconstructs the dependency structure from `bfo:BFO_0000062`
(`preceded_by`) on the step nodes.

In [5]:
combined = wf_graph + step_graph
print(f"Combined graph: {len(combined)} triples")
print()

SPARQL = """
PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?step_label ?pre_label
WHERE {
  ?step bfo:0000062 ?pre ;
        rdfs:label  ?step_label .
  ?pre  rdfs:label  ?pre_label .
}
ORDER BY ?pre_label ?step_label
"""

print(f'{"Step":<35} preceded by')
print("-" * 60)
for r in combined.query(SPARQL):
    print(f"{str(r.step_label):<35} {r.pre_label}")

Combined graph: 24 triples

Step                                preceded by
------------------------------------------------------------


Chemical Composition, batch A       Sample Preparation, batch A
Tensile Test, batch A               Sample Preparation, batch A
Hockett-Sherby Calibration, batch A Tensile Test, batch A


## Summary

| Where branching lives | How to express it |
|---|---|
| Workflow container | Ordered `steps` list — reflects intended execution order |
| Step records | `preceded_by` on each step — fan-out: same predecessor; fan-in: multiple predecessors |

The separation keeps the workflow schema simple while allowing arbitrary
DAG structures to be expressed in the step records.

---

**Next:** [Notebook 3](3_material_card_without_template.ipynb) builds a complete
cross-domain workflow from scratch: manufacturing → tensile test →
constitutive model calibration → FEM material card export.